In [1]:
import csv
import py2neo
from py2neo import Graph,Node,Relationship,NodeMatcher

In [ ]:
# 显式连接到 "neo4j" 数据库，而不是 "system"
g = Graph("bolt://localhost:7687", auth=("neo4j", "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"), name="neo4j")

In [3]:
# 删除所有节点和关系
g.run("MATCH (n) DETACH DELETE n")

(No data)

In [4]:
import csv

csv_path = r"C:\Users\ruyi\Documents\Codex\2026-09-06\source-pathway-aware-early-prediction-of\outputs\Cluster_0_triples.csv"

with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    rows = list(reader)    # 保存数据，供后面的单元格使用

print("列名：", header)
print("三元组数量：", len(rows))

for item in rows[:5]:
    head_name = item[0]       # Subject：公社名称
    relation_type = item[1]   # Predicate：关系
    tail_value = item[2]      # Object：属性值
    longitude = float(item[3])
    latitude = float(item[4])

    print(
        f"{head_name} -[{relation_type}]-> {tail_value}"
        f"  经度：{longitude}，纬度：{latitude}"
    )

列名： ['Subject', 'Predicate', 'Object', 'Longitude', 'Latitude']
三元组数量： 1764
Jiangdong -[Present]-> Jiangdong Subdistrict  经度：118.7284949，纬度：32.05109453
Jiangdong -[Elevation]-> 7  经度：118.7284949，纬度：32.05109453
Jiangdong -[Precipitation]-> 0.000  经度：118.7284949，纬度：32.05109453
Jiangdong -[Temperature]-> 288.706  经度：118.7284949，纬度：32.05109453
Jiangdong -[Wind]-> 3.695  经度：118.7284949，纬度：32.05109453


In [5]:
from neo4j import GraphDatabase
from getpass import getpass
import pandas as pd

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = getpass("请输入 Neo4j 密码：")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

def query(tx, cypher):
    result = tx.run(cypher)
    return pd.DataFrame(
        [record.data() for record in result],
        columns=result.keys()
    )

try:
    driver.verify_connectivity()
    print("Neo4j 连接成功")

    with driver.session(database="neo4j") as session:
        # 兼容不同版本的 Neo4j Python 驱动
        read = getattr(session, "execute_read", None)
        if read is None:
            read = session.read_transaction

        df_labels = read(query, """
            MATCH (n)
            RETURN labels(n) AS labels, count(*) AS node_count
            ORDER BY node_count DESC
        """)

        df_relations = read(query, """
            MATCH (:Commune)-[r]->()
            RETURN type(r) AS relation_type, count(*) AS relation_count
            ORDER BY relation_type
        """)

    print("\n节点标签：")
    display(df_labels)

    print("\n公社节点的关系类型：")
    display(df_relations)

finally:
    driver.close()

Neo4j 连接成功

节点标签：


,labels,node_count



公社节点的关系类型：


,relation_type,relation_count


In [6]:
import csv
from pathlib import Path
from neo4j import GraphDatabase

# 修改为你的 CSV 实际路径
csv_path = Path(
    r"C:\Users\ruyi\Documents\Codex\2026-09-06"
    r"\source-pathway-aware-early-prediction-of"
    r"\outputs\Cluster_0_triples.csv"
)

cluster_id = 0

# 读取并检查数据
with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    expected = {"Subject", "Predicate", "Object", "Longitude", "Latitude"}
    if not expected.issubset(reader.fieldnames or []):
        raise ValueError("CSV 列名不正确，请检查文件。")

    data = list(reader)

if not data:
    raise ValueError("CSV 没有数据。")

# 按属性分组，批量写入
groups = {}

for row in data:
    predicate = row["Predicate"].strip()

    # 经纬度不生成关系
    if predicate in {"Longitude", "Latitude"}:
        continue

    groups.setdefault(predicate, []).append({
        "subject": row["Subject"],
        "value": row["Object"],
        "longitude": float(row["Longitude"]),
        "latitude": float(row["Latitude"])
    })

def import_cluster(tx):
    for predicate, batch in groups.items():
        # 转义关系名称，使包含空格的英文属性也能使用
        relation_type = predicate.replace("`", "``")

        cypher = f"""
        UNWIND $rows AS row

        MERGE (c:Commune {{
            dataset: $dataset,
            cluster: $cluster,
            name: row.subject
        }})
        SET c.longitude = row.longitude,
            c.latitude = row.latitude,
            c.location = point({{
                longitude: row.longitude,
                latitude: row.latitude
            }})

        MERGE (a:Attribute {{
            dataset: $dataset,
            cluster: $cluster,
            attribute: $predicate,
            name: row.value
        }})

        MERGE (c)-[:`{relation_type}`]->(a)
        """

        tx.run(
            cypher,
            rows=batch,
            dataset="jiangsu_communes",
            cluster=cluster_id,
            predicate=predicate
        ).consume()

# 使用上一格中已填写的连接信息和密码
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

try:
    with driver.session(database="neo4j") as session:
        write = getattr(session, "execute_write", None)
        if write is None:
            write = session.write_transaction

        write(import_cluster)

        result = session.run("""
            MATCH (c:Commune {
                dataset: $dataset,
                cluster: $cluster
            })
            OPTIONAL MATCH (c)-[r]->(a:Attribute {
                dataset: $dataset,
                cluster: $cluster
            })
            RETURN count(DISTINCT c) AS communes,
                   count(r) AS relationships
        """,
            dataset="jiangsu_communes",
            cluster=cluster_id
        ).single()

        print(f"Cluster {cluster_id} 导入完成")
        print("公社节点数量：", result["communes"])
        print("关系数量：", result["relationships"])

finally:
    driver.close()

Cluster 0 导入完成
公社节点数量： 147
关系数量： 1764


In [ ]:
with open(r"E:\00论文\2025江苏人民公社\【submit1】heritage science\0609Chinese_knowledge_graph\0610people's commune.csv", 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头

    for row_num, item in enumerate(reader, start=2):
        head_name = item[0]
        tail_value = item[1]
        relation_type = item[2]
        location_value = item[3]

        # 创建实体节点（人民公社），标签设为 Commune，location 作为属性
        start_node = Node("Commune", name=head_name, location=location_value)

        # 创建属性节点（Attribute）
        end_node = Node("Attribute", name=tail_value)

        # 创建关系
        relation = Relationship(start_node, relation_type, end_node)

        # 合并节点和关系到图中
        g.merge(start_node, "Commune", "name")
        g.merge(end_node, "Attribute", "name")
        g.create(relation)  # 不使用 merge，因为关系没有主键约束

        print(f"已处理第 {row_num} 行：{head_name} -[{relation_type}]-> {tail_value}")

已处理第 2 行：紫金山 -[农业]-> 蔬菜
已处理第 3 行：紫金山 -[农业]-> 白菜
已处理第 4 行：紫金山 -[农业]-> 萝卜
已处理第 5 行：紫金山 -[农业]-> 黄瓜
已处理第 6 行：紫金山 -[专区]-> 南京市
已处理第 7 行：紫金山 -[市县]-> 南京市
已处理第 8 行：紫金山 -[现今]-> 孝陵卫街道
已处理第 9 行：紫金山 -[海拔]-> 海拔高
已处理第 10 行：紫金山 -[降雨]-> 降雨较少
已处理第 11 行：紫金山 -[气温]-> 气温高
已处理第 12 行：紫金山 -[风力]-> 风力强
已处理第 13 行：紫金山 -[至运河距离]-> 距离运河远
已处理第 14 行：紫金山 -[至铁路距离]-> 距离铁路近
已处理第 15 行：紫金山 -[至海洋距离]-> 距离海洋较远
已处理第 16 行：紫金山 -[至河流距离]-> 距离河流适中
已处理第 17 行：紫金山 -[至公路距离]-> 距离公路近
已处理第 18 行：竹箦 -[专区]-> 镇江专区
已处理第 19 行：竹箦 -[市县]-> 溧阳县
已处理第 20 行：竹箦 -[现今]-> 竹箦镇
已处理第 21 行：竹箦 -[海拔]-> 海拔适中
已处理第 22 行：竹箦 -[降雨]-> 降雨较多
已处理第 23 行：竹箦 -[气温]-> 气温高
已处理第 24 行：竹箦 -[风力]-> 风力较强
已处理第 25 行：竹箦 -[至运河距离]-> 距离运河较近
已处理第 26 行：竹箦 -[至铁路距离]-> 距离铁路远
已处理第 27 行：竹箦 -[至海洋距离]-> 距离海洋较远
已处理第 28 行：竹箦 -[至河流距离]-> 距离河流适中
已处理第 29 行：竹箦 -[至公路距离]-> 距离公路近
已处理第 30 行：朱集 -[农业]-> 麦
已处理第 31 行：朱集 -[农业]-> 小麦
已处理第 32 行：朱集 -[专区]-> 徐州专区
已处理第 33 行：朱集 -[市县]-> 睢宁县
已处理第 34 行：朱集 -[现今]-> 朱集村
已处理第 35 行：朱集 -[海拔]-> 海拔高
已处理第 36 行：朱集 -[降雨]-> 降雨较多
已处理第 37 行：朱集 -[气温]-> 气温较低
已处理第 38 行：朱集 -[风力]-> 风力弱
已处理第 39 行

In [6]:
from neo4j import GraphDatabase
import pandas as pd

# Neo4j 配置
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"

# 建立连接
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# 查询封装函数
def query(tx, cypher):
    result = tx.run(cypher)
    return pd.DataFrame([r.data() for r in result])

with driver.session() as session:
    # 1. 基本信息：name, location
    df_base = session.read_transaction(query, """
        MATCH (c:Commune)
        RETURN c.name AS name, c.location AS location
    """)

    # 2. 农业产业属性
    df_agri = session.read_transaction(query, """
        MATCH (c:Commune)-[:农业]->(a:Attribute)
        RETURN c.name AS name, collect(a.name) AS agri_attrs
    """)

    # 3. 工业产业属性
    df_ind = session.read_transaction(query, """
        MATCH (c:Commune)-[:工业]->(a:Attribute)
        RETURN c.name AS name, collect(a.name) AS ind_attrs
    """)

    # 4. 其他产业
    df_other = session.read_transaction(query, """
        MATCH (c:Commune)-[:其他]->(a:Attribute)
        RETURN c.name AS name, collect(a.name) AS other_attrs
    """)

    # 5. 所在专区
    df_zone = session.read_transaction(query, """
        MATCH (c:Commune)-[:专区]->(a:Attribute)
        RETURN c.name AS name, a.name AS zone
    """)

    # 6. 所在市县
    df_county = session.read_transaction(query, """
        MATCH (c:Commune)-[:市县]->(a:Attribute)
        RETURN c.name AS name, a.name AS county
    """)

    # 7. 现今归属
    df_now = session.read_transaction(query, """
        MATCH (c:Commune)-[:现今]->(a:Attribute)
        RETURN c.name AS name, a.name AS now_admin
    """)

    # 8. 自然环境类（海拔、降雨、气温、风力、距离）
    df_env = {}
    env_rels = {
        "海拔": "altitude",
        "降雨": "rain",
        "气温": "temp",
        "风力": "wind",
        "至公路距离": "dist_road",
        "至铁路距离": "dist_rail",
        "至运河距离": "dist_canal",
        "至海洋距离": "dist_ocean",
        "至河流距离": "dist_river"
    }

    for rel, col in env_rels.items():
        cypher = f"""
            MATCH (c:Commune)-[:`{rel}`]->(a:Attribute)
            RETURN c.name AS name, a.name AS {col}
        """
        df_env[col] = session.read_transaction(query, cypher)

# 合并所有 DataFrame
df_all = df_base.copy()
for df in [df_agri, df_ind, df_other, df_zone, df_county, df_now] + list(df_env.values()):
    df_all = pd.merge(df_all, df, on="name", how="left")

# 缺失值填充
df_all.fillna("", inplace=True)

# 保存为 CSV
df_all.to_csv(r"E:\【论文】\heritage science\0609Chinese_knowledge_graph\commune_features.csv", index=False, encoding="utf-8-sig")

print("✅ 导出完成：commune_features.csv")

C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:19: DeprecationWarning: read_transaction has been renamed to execute_read
  df_base = session.read_transaction(query, """
C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:25: DeprecationWarning: read_transaction has been renamed to execute_read
  df_agri = session.read_transaction(query, """
C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:31: DeprecationWarning: read_transaction has been renamed to execute_read
  df_ind = session.read_transaction(query, """
C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:37: DeprecationWarning: read_transaction has been renamed to execute_read
  df_other = session.read_transaction(query, """
C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:43: DeprecationWarning: read_transaction has been renamed to execute_read
  df_zone = session.read_transaction(query, """
C:\Users\ruyi\AppData\Local\Temp\ipykernel_25116\1625505844.py:49: Deprecat

✅ 导出完成：commune_features.csv


In [7]:
import re

def extract_lat_lon(location):
    match = re.search(r'\(?([0-9.]+),\s*([0-9.]+)\)?', str(location))
    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        return pd.Series({'lat': lat, 'lon': lon})
    return pd.Series({'lat': None, 'lon': None})

df_all[['lat', 'lon']] = df_all['location'].apply(extract_lat_lon)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# 对农业、工业、其他三列都做one-hot编码
mlb_agri = MultiLabelBinarizer()
agri_df = pd.DataFrame(mlb_agri.fit_transform(df_all['agri_attrs']), 
                       columns=[f"agri_{c}" for c in mlb_agri.classes_])
mlb_ind = MultiLabelBinarizer()
ind_df = pd.DataFrame(mlb_ind.fit_transform(df_all['ind_attrs']), 
                      columns=[f"ind_{c}" for c in mlb_ind.classes_])
mlb_other = MultiLabelBinarizer()
other_df = pd.DataFrame(mlb_other.fit_transform(df_all['other_attrs']), 
                        columns=[f"other_{c}" for c in mlb_other.classes_])

# 合并
df_all = pd.concat([df_all, agri_df, ind_df, other_df], axis=1)

In [3]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. 读取原始数据
# ============================================================

input_file = r"E:\00论文\2025江苏人民公社\【submit1】heritage science\English_knowledge_graph\processed_data.csv"

df = pd.read_csv(input_file)

print("数据维度：", df.shape)
print("字段：")
print(df.columns.tolist())


# ============================================================
# 2. 基础字段
# ============================================================

HEAD_COL = "head"

# 三个需要分别建立知识图谱的类别
CATEGORY_COLS = [
    "Industry",
    "industry",
    "other"
]


# ============================================================
# 3. 公社的基本信息
#    这些信息不作为三个类别本身，
#    但可以作为 Commune 节点的属性
# ============================================================

BASE_COLUMNS = [
    "district",
    "city",
    "now",
    "lon",
    "lat"
]


# ============================================================
# 4. 环境变量
#
# 原始环境变量 + 分级后的环境变量全部保留
# ============================================================

ENV_COLUMNS = [
    # 原始环境变量
    "elev",
    "rain",
    "temp",
    "wind",
    "dist_canal",
    "dist_rail",
    "dist_ocean",
    "dist_river",
    "dist_road",

    # 分级变量
    "rain_level",
    "temp_level",
    "wind_level",
    "dist_canal_level",
    "dist_rail_level",
    "dist_ocean_level",
    "dist_river_level",
    "dist_road_level"
]


# ============================================================
# 5. 检查字段是否存在
# ============================================================

required_columns = [HEAD_COL] + CATEGORY_COLS + BASE_COLUMNS + ENV_COLUMNS

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"以下字段在原始数据中不存在：{missing_columns}"
    )

print("\n字段检查通过。")


# ============================================================
# 6. 定义关系名称
# ============================================================

RELATION_MAP = {

    # 三个类别
    "Industry": "HAS_INDUSTRY",
    "industry": "HAS_INDUSTRY_ATTRIBUTE",
    "other": "HAS_OTHER",

    # 基本信息
    "district": "LOCATED_IN_DISTRICT",
    "city": "LOCATED_IN_CITY",
    "now": "CURRENT_LOCATION",

    # 环境
    "elev": "HAS_ELEVATION",
    "rain": "HAS_RAIN",
    "temp": "HAS_TEMPERATURE",
    "wind": "HAS_WIND",
    "dist_canal": "DISTANCE_TO_CANAL",
    "dist_rail": "DISTANCE_TO_RAIL",
    "dist_ocean": "DISTANCE_TO_OCEAN",
    "dist_river": "DISTANCE_TO_RIVER",
    "dist_road": "DISTANCE_TO_ROAD",

    # 环境等级
    "rain_level": "HAS_RAIN_LEVEL",
    "temp_level": "HAS_TEMPERATURE_LEVEL",
    "wind_level": "HAS_WIND_LEVEL",
    "dist_canal_level": "HAS_DIST_CANAL_LEVEL",
    "dist_rail_level": "HAS_DIST_RAIL_LEVEL",
    "dist_ocean_level": "HAS_DIST_OCEAN_LEVEL",
    "dist_river_level": "HAS_DIST_RIVER_LEVEL",
    "dist_road_level": "HAS_DIST_ROAD_LEVEL"
}


# ============================================================
# 7. 辅助函数
# ============================================================

def is_valid_value(value):
    """
    判断单元格是否有有效内容。
    注意：
    0 是有效值，不能被删除。
    """

    if pd.isna(value):
        return False

    if isinstance(value, str):
        return value.strip() != ""

    return True


# ============================================================
# 8. 构建三元组
# ============================================================

def create_triples(df, category_column):

    triples = []

    for _, row in df.iterrows():

        head = row[HEAD_COL]

        if not is_valid_value(head):
            continue

        # ----------------------------------------------------
        # A. 类别属性
        # ----------------------------------------------------

        value = row[category_column]

        if is_valid_value(value):

            triples.append({
                "head": str(head),
                "relation": RELATION_MAP[category_column],
                "tail": str(value)
            })


        # ----------------------------------------------------
        # B. 环境变量
        # ----------------------------------------------------

        for col in ENV_COLUMNS:

            value = row[col]

            if is_valid_value(value):

                triples.append({
                    "head": str(head),
                    "relation": RELATION_MAP[col],
                    "tail": str(value)
                })


    return pd.DataFrame(triples)


# ============================================================
# 9. 分别生成三个三元组数据集
# ============================================================

output_dir = r"E:\00论文\2025江苏人民公社\JAABE返修2\knowledge_graph_triples"

os.makedirs(output_dir, exist_ok=True)


for category in CATEGORY_COLS:

    triples_df = create_triples(
        df,
        category
    )

    output_file = os.path.join(
        output_dir,
        f"{category}_triples.csv"
    )

    triples_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\n{category} knowledge graph"
        f"\n三元组数量：{len(triples_df)}"
        f"\n保存位置：{output_file}"
    )

数据维度： (334, 26)
字段：
['district', 'city', 'head', 'now', 'lon', 'lat', 'Industry', 'industry', 'other', 'elev', 'rain', 'temp', 'wind', 'dist_canal', 'dist_rail', 'dist_ocean', 'dist_river', 'dist_road', 'rain_level', 'temp_level', 'wind_level', 'dist_canal_level', 'dist_rail_level', 'dist_ocean_level', 'dist_river_level', 'dist_road_level']

字段检查通过。

Industry knowledge graph
三元组数量：5937
保存位置：E:\00论文\2025江苏人民公社\JAABE返修2\knowledge_graph_triples\Industry_triples.csv

industry knowledge graph
三元组数量：5723
保存位置：E:\00论文\2025江苏人民公社\JAABE返修2\knowledge_graph_triples\industry_triples.csv

other knowledge graph
三元组数量：5806
保存位置：E:\00论文\2025江苏人民公社\JAABE返修2\knowledge_graph_triples\other_triples.csv


In [7]:
import csv
from pathlib import Path
from getpass import getpass
from neo4j import GraphDatabase

# 1. CSV 所在文件夹：按你的实际位置修改
DATA_DIR = Path(
    r"C:\Users\ruyi\Documents\Codex\2026-09-06"
    r"\source-pathway-aware-early-prediction-of\outputs"
)

# 2. Neo4j 连接配置
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = getpass("请输入 Neo4j 密码：")

DATASET = "jiangsu_communes"

# 允许导入的属性
PREDICATES = {
    "Present", "Elevation", "Precipitation", "Temperature", "Wind",
    "Distance to canal", "Distance to railway", "Distance to ocean",
    "Distance to river", "Distance to road",
    "Distance to central city", "cluster"
}

# 3. 先读取并检查两个文件
all_groups = {}

for cluster_id in (1, 2):
    csv_path = DATA_DIR / f"Cluster_{cluster_id}_triples.csv"

    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        required = {
            "Subject", "Predicate", "Object",
            "Longitude", "Latitude"
        }

        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"{csv_path.name} 的列名不正确")

        data = list(reader)

    if not data:
        raise ValueError(f"{csv_path.name} 没有数据")

    groups = {}

    for line, row in enumerate(data, start=2):
        predicate = row["Predicate"].strip()

        if predicate not in PREDICATES:
            raise ValueError(
                f"{csv_path.name} 第 {line} 行属性不正确：{predicate}"
            )

        if predicate == "cluster":
            if float(row["Object"]) != cluster_id:
                raise ValueError(f"{csv_path.name} 中的 cluster 不一致")

        longitude = float(row["Longitude"])
        latitude = float(row["Latitude"])

        if not (-180 <= longitude <= 180 and -90 <= latitude <= 90):
            raise ValueError(f"{csv_path.name} 第 {line} 行坐标无效")

        groups.setdefault(predicate, []).append({
            "subject": row["Subject"],
            "value": row["Object"],
            "longitude": longitude,
            "latitude": latitude
        })

    all_groups[cluster_id] = groups

    print(f"{csv_path.name}：读取 {len(data)} 条三元组")


# 4. 批量导入函数
def import_cluster(tx, cluster_id, groups):
    for predicate, batch in groups.items():
        relation_type = predicate.replace("`", "``")

        cypher = f"""
        UNWIND $rows AS row

        MERGE (c:Commune {{
            dataset: $dataset,
            cluster: $cluster,
            name: row.subject
        }})
        SET c.longitude = row.longitude,
            c.latitude = row.latitude,
            c.location = point({{
                longitude: row.longitude,
                latitude: row.latitude
            }})

        MERGE (a:Attribute {{
            dataset: $dataset,
            cluster: $cluster,
            attribute: $predicate,
            name: row.value
        }})

        MERGE (c)-[:`{relation_type}`]->(a)
        """

        tx.run(
            cypher,
            rows=batch,
            dataset=DATASET,
            cluster=cluster_id,
            predicate=predicate
        ).consume()


# 5. 连接数据库，依次导入并查询数量
with GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
) as driver:

    driver.verify_connectivity()
    print("\nNeo4j 连接成功")

    with driver.session(database="neo4j") as session:
        write = getattr(session, "execute_write", None)
        if write is None:
            write = session.write_transaction

        for cluster_id in (1, 2):
            write(
                import_cluster,
                cluster_id,
                all_groups[cluster_id]
            )

            result = session.run("""
                MATCH (c:Commune {
                    dataset: $dataset,
                    cluster: $cluster
                })
                OPTIONAL MATCH (c)-[r]->(a:Attribute {
                    dataset: $dataset,
                    cluster: $cluster
                })
                RETURN count(DISTINCT c) AS communes,
                       count(r) AS relationships
            """,
                dataset=DATASET,
                cluster=cluster_id
            ).single()

            print(f"\nCluster {cluster_id} 导入完成")
            print("公社节点数量：", result["communes"])
            print("关系数量：", result["relationships"])

Cluster_1_triples.csv：读取 588 条三元组
Cluster_2_triples.csv：读取 1116 条三元组

Neo4j 连接成功

Cluster 1 导入完成
公社节点数量： 49
关系数量： 588

Cluster 2 导入完成
公社节点数量： 93
关系数量： 1116


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. 文件路径
# ============================================================

INPUT_FILE = Path(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\English_processed_data0527.csv"
)

OUTPUT_DIR = Path(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\three_commune_graphs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. 读取原始数据
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("数据维度：", df.shape)
print("字段：")
print(df.columns.tolist())


# ============================================================
# 3. 需要分别构建知识图谱的三个公社
#
# 每个公社 = 一张独立知识图谱
# ============================================================

TARGET_COMMUNES = [
    "Xinqiao",
    "Zhangqiao",
    "Dingji"
]


# ============================================================
# 4. 字段与关系名称
# ============================================================

RELATION_MAP = {

    # --------------------
    # 基本信息
    # --------------------
    "district": "LOCATED_IN_DISTRICT",
    "city": "LOCATED_IN_CITY",
    "now": "CURRENT_LOCATION",

    # --------------------
    # 三类信息
    # --------------------
    "Industry": "AGRICULTURAL",
    "industry": "INDUSTRIAL",
    "other": "OTHER",

    # --------------------
    # 环境变量
    # --------------------
    "elev": "HAS_ELEVATION",

    "rain_level": "HAS_RAIN_LEVEL",
    "temp_level": "HAS_TEMPERATURE_LEVEL",
    "wind_level": "HAS_WIND_LEVEL",

    "dist_canal_level": "DISTANCE_TO_CANAL",
    "dist_rail_level": "DISTANCE_TO_RAIL",
    "dist_ocean_level": "DISTANCE_TO_OCEAN",
    "dist_river_level": "DISTANCE_TO_RIVER",
    "dist_road_level": "DISTANCE_TO_ROAD"
}


# ============================================================
# 5. 检查必须存在的字段
# ============================================================

required_columns = [
    "head",
    "lon",
    "lat"
] + list(RELATION_MAP.keys())

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"以下字段在 CSV 中不存在：{missing_columns}"
    )

print("\n字段检查通过。")


# ============================================================
# 6. 判断是否为有效值
# ============================================================

def is_valid_value(value):
    """
    判断值是否有效。
    注意：0 是有效值。
    """

    if pd.isna(value):
        return False

    if isinstance(value, str):
        return value.strip() != ""

    return True


# ============================================================
# 7. 数字统一转换为三位小数
# ============================================================

def format_value(value):
    """
    数值统一保留三位小数。

    例如：
    5          -> 5.000
    0          -> 0.000
    0.040619   -> 0.041
    15011.7002 -> 15011.700

    普通文本保持原样。
    """

    if not is_valid_value(value):
        return None

    # 原本就是数值
    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):
        return f"{float(value):.3f}"

    text = str(value).strip()

    # 如果字符串整体也是数字
    try:
        number = float(text)
        return f"{number:.3f}"

    except ValueError:
        return text


# ============================================================
# 8. 创建单个公社的知识图谱三元组
# ============================================================

def create_commune_graph(row):

    commune = str(row["head"]).strip()

    triples = []

    # --------------------------------------------------------
    # 遍历所有需要进入知识图谱的字段
    # --------------------------------------------------------

    for column, relation in RELATION_MAP.items():

        value = row[column]

        if not is_valid_value(value):
            continue


        # ====================================================
        # Agricultural / Industrial
        #
        # 如果一个单元格中存在多个项目：
        #
        # Pig farming, corn, hemp
        #
        # 则拆成：
        #
        # Dingji -> AGRICULTURAL -> Pig farming
        # Dingji -> AGRICULTURAL -> corn
        # Dingji -> AGRICULTURAL -> hemp
        # ====================================================

        if column in ["Industry", "industry"]:

            items = [
                item.strip()
                for item in str(value).split(",")
                if item.strip()
            ]

            for item in items:

                triples.append({
                    "Subject": commune,
                    "Predicate": relation,
                    "Object": format_value(item),
                    "Longitude": format_value(row["lon"]),
                    "Latitude": format_value(row["lat"])
                })


        # ====================================================
        # 其他字段
        # ====================================================

        else:

            triples.append({
                "Subject": commune,
                "Predicate": relation,
                "Object": format_value(value),
                "Longitude": format_value(row["lon"]),
                "Latitude": format_value(row["lat"])
            })


    return pd.DataFrame(triples)


# ============================================================
# 9. 分别构建三个独立知识图谱
# ============================================================

for commune in TARGET_COMMUNES:

    # 忽略大小写匹配
    matched = df[
        df["head"]
        .astype(str)
        .str.strip()
        .str.lower()
        == commune.lower()
    ]


    # --------------------------------------------------------
    # 没找到
    # --------------------------------------------------------

    if matched.empty:

        print("\n" + "=" * 60)
        print(f"[未找到] {commune}")
        print("=" * 60)

        continue


    # --------------------------------------------------------
    # 出现多个同名公社
    # --------------------------------------------------------

    if len(matched) > 1:

        print("\n" + "=" * 60)
        print(
            f"[警告] {commune} 出现 {len(matched)} 条记录"
        )

        print(
            matched[
                ["district", "city", "head", "now"]
            ].to_string(index=False)
        )

        print("=" * 60)

        continue


    # --------------------------------------------------------
    # 唯一匹配
    # --------------------------------------------------------

    row = matched.iloc[0]

    graph_df = create_commune_graph(row)


    # --------------------------------------------------------
    # 每个公社单独保存一个 CSV
    # --------------------------------------------------------

    output_file = (
        OUTPUT_DIR /
        f"{commune}_knowledge_graph.csv"
    )

    graph_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )


    # --------------------------------------------------------
    # 输出检查
    # --------------------------------------------------------

    print("\n" + "=" * 60)

    print(f"知识图谱：{commune}")

    print(
        f"位置：{row['district']} / "
        f"{row['city']}"
    )

    print(
        f"三元组数量：{len(graph_df)}"
    )

    print(
        f"保存位置：{output_file}"
    )

    print("-" * 60)

    print(
        graph_df.to_string(index=False)
    )

    print("=" * 60)


# ============================================================
# 10. 完成
# ============================================================

print("\n三个公社的知识图谱三元组处理完成。")

数据维度： (334, 18)
字段：
['district', 'city', 'head', 'now', 'lon', 'lat', 'Industry', 'industry', 'other', 'elev', 'rain_level', 'temp_level', 'wind_level', 'dist_canal_level', 'dist_rail_level', 'dist_ocean_level', 'dist_river_level', 'dist_road_level']

字段检查通过。

[未找到] Xinqiao

知识图谱：Zhangqiao
位置：Suzhou District / Wuxian County
三元组数量：14
保存位置：E:\00论文\2025江苏人民公社\【submit1】heritage science\English_knowledge_graph\three_commune_graphs\Zhangqiao_knowledge_graph.csv
------------------------------------------------------------
  Subject             Predicate                       Object Longitude Latitude
Zhangqiao   LOCATED_IN_DISTRICT              Suzhou District   120.064   32.115
Zhangqiao       LOCATED_IN_CITY                Wuxian County   120.064   32.115
Zhangqiao      CURRENT_LOCATION               Zhangqiao Town   120.064   32.115
Zhangqiao          AGRICULTURAL                  Sericulture   120.064   32.115
Zhangqiao                 OTHER Production team 11 and above   120.064   32.115

In [3]:
import pandas as pd

file_path = (
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\English_processed_data0527.csv"
)

df = pd.read_csv(file_path)

# ============================================================
# 1. 在所有字段中搜索 Xinqiao
# ============================================================

mask = df.astype(str).apply(
    lambda col: col.str.contains(
        "Xinqiao",
        case=False,
        na=False
    )
).any(axis=1)

result = df[mask]

print("包含 Xinqiao 的记录数：", len(result))

if len(result) > 0:
    print(result.to_string())
else:
    print("整个 CSV 中都没有出现 Xinqiao")

包含 Xinqiao 的记录数： 2
              district              city        head           now         lon        lat Industry industry                                          other  elev rain_level temp_level wind_level dist_canal_level dist_rail_level dist_ocean_level dist_river_level dist_road_level
22  Zhenjiang District    Danyang County  New Bridge  Xinqiao Town  119.833290  32.082453               NaN                                            NaN     4     Higher     Higher     Higher           Medium           Lower           Higher              Low            High
83   Yangzhou District  Jingjiang County  New Bridge  Xinqiao Town  120.095204  32.001610    Wheat      NaN  Early-maturing wheat planting area of 6000 mu     5     Medium     Medium     Medium            Lower          Medium           Medium              Low          Medium


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. 文件路径
# ============================================================

INPUT_FILE = Path(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\English_processed_data0527.csv"
)

OUTPUT_DIR = Path(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\three_commune_graphs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. 读取数据
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("数据维度：", df.shape)


# ============================================================
# 3. 指定三个公社
#
# Xinqiao:
# CSV index = 83
# head = New Bridge
# now = Xinqiao Town
#
# Zhangqiao / Dingji:
# 直接按照 head 匹配
# ============================================================

TARGETS = {
    "Xinqiao": {
        "mode": "index",
        "value": 83
    },

    "Zhangqiao": {
        "mode": "head",
        "value": "Zhangqiao"
    },

    "Dingji": {
        "mode": "head",
        "value": "Dingji"
    }
}


# ============================================================
# 4. 关系名称
# ============================================================

RELATION_MAP = {

    # 基本信息
    "district": "LOCATED_IN_DISTRICT",
    "city": "LOCATED_IN_CITY",
    "now": "CURRENT_LOCATION",

    # 农业 / 工业 / 其他
    "Industry": "AGRICULTURAL",
    "industry": "INDUSTRIAL",
    "other": "OTHER",

    # 环境
    "elev": "HAS_ELEVATION",

    "rain_level": "HAS_RAIN_LEVEL",
    "temp_level": "HAS_TEMPERATURE_LEVEL",
    "wind_level": "HAS_WIND_LEVEL",

    "dist_canal_level": "DISTANCE_TO_CANAL",
    "dist_rail_level": "DISTANCE_TO_RAIL",
    "dist_ocean_level": "DISTANCE_TO_OCEAN",
    "dist_river_level": "DISTANCE_TO_RIVER",
    "dist_road_level": "DISTANCE_TO_ROAD"
}


# ============================================================
# 5. 判断有效值
# ============================================================

def is_valid(value):

    if pd.isna(value):
        return False

    if isinstance(value, str):
        return value.strip() != ""

    return True


# ============================================================
# 6. 数值统一三位小数
# ============================================================

def format_value(value):

    if not is_valid(value):
        return None

    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):
        return f"{float(value):.3f}"

    text = str(value).strip()

    try:
        return f"{float(text):.3f}"
    except ValueError:
        return text


# ============================================================
# 7. 拆分 Agricultural / Industrial
# ============================================================

def split_items(value):

    if not is_valid(value):
        return []

    text = str(value).strip()

    # 英文逗号拆分
    items = [
        x.strip()
        for x in text.split(",")
        if x.strip()
    ]

    return items


# ============================================================
# 8. 构建一个独立知识图谱
# ============================================================

def build_graph(row, graph_name):

    triples = []

    for column, relation in RELATION_MAP.items():

        value = row[column]

        if not is_valid(value):
            continue

        # ----------------------------------------------------
        # Agricultural / Industrial 可以包含多个项目
        # ----------------------------------------------------

        if column in ["Industry", "industry"]:

            items = split_items(value)

            for item in items:

                triples.append({
                    "Subject": graph_name,
                    "Predicate": relation,
                    "Object": format_value(item),
                    "Longitude": format_value(row["lon"]),
                    "Latitude": format_value(row["lat"])
                })

        # ----------------------------------------------------
        # 其他属性
        # ----------------------------------------------------

        else:

            triples.append({
                "Subject": graph_name,
                "Predicate": relation,
                "Object": format_value(value),
                "Longitude": format_value(row["lon"]),
                "Latitude": format_value(row["lat"])
            })

    return pd.DataFrame(triples)


# ============================================================
# 9. 分别提取三个公社
# ============================================================

for graph_name, config in TARGETS.items():

    # --------------------------------------------------------
    # Xinqiao：按照 index = 83 精确选择
    # --------------------------------------------------------

    if config["mode"] == "index":

        idx = config["value"]

        if idx not in df.index:
            raise ValueError(
                f"{graph_name}: index {idx} 不存在"
            )

        row = df.loc[idx]


    # --------------------------------------------------------
    # Zhangqiao / Dingji：按照 head 精确选择
    # --------------------------------------------------------

    elif config["mode"] == "head":

        target_head = config["value"]

        matched = df[
            df["head"]
            .astype(str)
            .str.strip()
            .str.lower()
            == target_head.lower()
        ]

        if len(matched) == 0:
            raise ValueError(
                f"找不到 {graph_name}"
            )

        if len(matched) > 1:
            raise ValueError(
                f"{graph_name} 存在多个同名记录：\n"
                f"{matched[['district', 'city', 'head', 'now']]}"
            )

        row = matched.iloc[0]

    else:
        raise ValueError("未知匹配模式")


    # ========================================================
    # 10. 显示实际选择的数据
    # ========================================================

    print("\n" + "=" * 70)
    print(f"正在构建：{graph_name}")
    print("=" * 70)

    print("原始 head：", row["head"])
    print("now：", row["now"])
    print("district：", row["district"])
    print("city：", row["city"])
    print("Industry：", row["Industry"])
    print("industry：", row["industry"])
    print("other：", row["other"])
    print("elev：", row["elev"])


    # ========================================================
    # 11. 构建独立三元组
    # ========================================================

    graph_df = build_graph(
        row=row,
        graph_name=graph_name
    )


    # ========================================================
    # 12. 分别保存
    # ========================================================

    output_file = (
        OUTPUT_DIR /
        f"{graph_name}_knowledge_graph.csv"
    )

    graph_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n生成的三元组：")
    print(graph_df.to_string(index=False))

    print(
        f"\n保存完成：{output_file}"
    )


print("\n三个独立知识图谱数据生成完成。")

数据维度： (334, 18)

正在构建：Xinqiao
原始 head： New Bridge
now： Xinqiao Town
district： Yangzhou District
city： Jingjiang County
Industry： Wheat
industry： nan
other： Early-maturing wheat planting area of 6000 mu
elev： 5

生成的三元组：
Subject             Predicate                                        Object Longitude Latitude
Xinqiao   LOCATED_IN_DISTRICT                             Yangzhou District   120.095   32.002
Xinqiao       LOCATED_IN_CITY                              Jingjiang County   120.095   32.002
Xinqiao      CURRENT_LOCATION                                  Xinqiao Town   120.095   32.002
Xinqiao          AGRICULTURAL                                         Wheat   120.095   32.002
Xinqiao                 OTHER Early-maturing wheat planting area of 6000 mu   120.095   32.002
Xinqiao         HAS_ELEVATION                                         5.000   120.095   32.002
Xinqiao        HAS_RAIN_LEVEL                                        Medium   120.095   32.002
Xinqiao HAS_TEMPERATU

In [5]:
from py2neo import Graph, Node, Relationship

# 连接 Neo4j
g = Graph(
    "bolt://localhost:7687",
    auth=("neo4j", "12345678"),
    name="neo4j"
)

print("Neo4j 连接成功")

Neo4j 连接成功


In [6]:
import csv
from py2neo import Graph, Node, Relationship


# ============================================================
# 1. Neo4j 连接
# ============================================================

g = Graph(
    "bolt://localhost:7687",
    auth=("neo4j", "12345678"),
    name="neo4j"
)


# ============================================================
# 2. Xinqiao CSV
# ============================================================

csv_path = (
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\three_commune_graphs"
    r"\Xinqiao_knowledge_graph.csv"
)


# ============================================================
# 3. 读取 CSV
# ============================================================

with open(
    csv_path,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.DictReader(f)
    rows = list(reader)


print("读取三元组数量：", len(rows))


# ============================================================
# 4. 删除之前的 Xinqiao 图
#
# 注意：
# 这里只删除 Xinqiao
# 不要再使用 MATCH (n) DETACH DELETE n
# 否则其他知识图谱也会被删除
# ============================================================

g.run("""
MATCH (n {graph_id: 'Xinqiao'})
DETACH DELETE n
""")

print("旧 Xinqiao 图已清除")


# ============================================================
# 5. 创建 Xinqiao 中心节点
# ============================================================

commune = Node(
    "Commune",
    name="Xinqiao",
    graph_id="Xinqiao"
)

g.create(commune)


# ============================================================
# 6. 创建属性节点和关系
# ============================================================

for row in rows:

    subject = row["Subject"].strip()
    predicate = row["Predicate"].strip()
    obj = row["Object"].strip()

    longitude = float(row["Longitude"])
    latitude = float(row["Latitude"])


    # --------------------------------------------------------
    # 给中心公社节点添加经纬度
    # --------------------------------------------------------

    commune["longitude"] = longitude
    commune["latitude"] = latitude

    g.push(commune)


    # --------------------------------------------------------
    # 创建属性节点
    # --------------------------------------------------------

    attribute = Node(
        "Attribute",
        name=obj,
        graph_id="Xinqiao",
        attribute_type=predicate
    )

    g.create(attribute)


    # --------------------------------------------------------
    # 创建关系
    #
    # 例如：
    # Xinqiao -[:AGRICULTURAL]-> Wheat
    #
    # Xinqiao -[:HAS_ELEVATION]-> 5.000
    # --------------------------------------------------------

    relationship = Relationship(
        commune,
        predicate,
        attribute
    )

    g.create(relationship)


print("Xinqiao 知识图谱导入完成！")

读取三元组数量： 14
旧 Xinqiao 图已清除
Xinqiao 知识图谱导入完成！


In [7]:
import csv
from py2neo import Graph, Node, Relationship


# ============================================================
# 1. Neo4j
# ============================================================

g = Graph(
    "bolt://localhost:7687",
    auth=("neo4j", "12345678"),
    name="neo4j"
)


# ============================================================
# 2. 通用导入函数
# ============================================================

def import_knowledge_graph(csv_path, graph_name):

    # --------------------------------------------------------
    # 读取 CSV
    # --------------------------------------------------------

    with open(
        csv_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.DictReader(f)
        rows = list(reader)


    if not rows:
        raise ValueError(
            f"{graph_name} CSV 没有数据"
        )


    print(
        f"{graph_name}：读取 {len(rows)} 条三元组"
    )


    # --------------------------------------------------------
    # 只删除当前这一张图
    # --------------------------------------------------------

    g.run(
        """
        MATCH (n {graph_id: $graph_id})
        DETACH DELETE n
        """,
        graph_id=graph_name
    )


    # --------------------------------------------------------
    # 中心节点
    # --------------------------------------------------------

    longitude = float(rows[0]["Longitude"])
    latitude = float(rows[0]["Latitude"])

    commune = Node(
        "Commune",

        name=graph_name,
        graph_id=graph_name,

        longitude=longitude,
        latitude=latitude
    )

    g.create(commune)


    # --------------------------------------------------------
    # 三元组
    # --------------------------------------------------------

    for row in rows:

        predicate = row["Predicate"].strip()
        obj = row["Object"].strip()


        # 创建属性节点
        attribute = Node(
            "Attribute",

            name=obj,
            graph_id=graph_name,
            attribute_type=predicate
        )

        g.create(attribute)


        # 创建真实关系
        relationship = Relationship(
            commune,
            predicate,
            attribute
        )

        g.create(relationship)


    print(
        f"{graph_name} 知识图谱导入完成"
    )

In [8]:
import_knowledge_graph(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\three_commune_graphs"
    r"\Zhangqiao_knowledge_graph.csv",

    "Zhangqiao"
)

Zhangqiao：读取 14 条三元组
Zhangqiao 知识图谱导入完成


In [9]:
import_knowledge_graph(
    r"E:\00论文\2025江苏人民公社\【submit1】heritage science"
    r"\English_knowledge_graph\three_commune_graphs"
    r"\Dingji_knowledge_graph.csv",

    "Dingji"
)

Dingji：读取 17 条三元组
Dingji 知识图谱导入完成
